# IBM SkillsBuild Data Analytics with AI Internship: Final Project
# Customer Churn Prediction & Risk Analytics Using E-Commerce Data

**Author / Intern:** Samiksha Rajendra Nandurkar  
**Program:** AICTE - IBM SkillsBuild Internship in Data Analytics with AI  
**Masterclass Track:** Masterclass 3 — *"AI Predicts What Happens Next"*  
**Technologies:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, OpenPyXL  

---

### Project Overview
In modern e-commerce, customer retention is vastly more cost-effective than new customer acquisition. Acquiring a new customer can cost 5 to 7 times more than retaining an existing one. This project implements an end-to-end predictive analytics and machine learning solution to identify at-risk customers before they defect, quantify their lapse probability, and design targeted retention interventions.


## 1. Business Problem

An e-commerce enterprise observes gradual customer attrition, where repeat purchasers diminish order frequency and transition into inactivity. 

Key business challenges include:
1. **Lack of Early Warning Signals:** Customer disengagement is typically noticed only after months of complete dormancy.
2. **Untargeted Retention Campaigns:** Blasting generic discount codes across the entire customer base erodes margins and produces poor ROI.
3. **High-Value Account Vulnerability:** Losing top-tier spenders severely impacts quarterly revenue.

The core business objective is to transition from **reactive customer recovery** to **proactive churn intervention** by predicting the probability of customer lapse using machine learning.


## 2. Project Objectives

Following the IBM SkillsBuild Masterclass 3 methodology, this project executes the following milestones:
- **Data Ingestion & Hygiene:** Cleanse raw transactional logs, remove anomalies, and parse mixed-format timestamps.
- **Exploratory Data Analysis (EDA):** Identify revenue distributions, order trends, seasonality, and spending concentration.
- **Customer Aggregation & RFM Modeling:** Aggregate transactions to a single row per customer and engineer **Recency, Frequency, and Monetary (RFM)** behavioral metrics.
- **Empirical Churn Definition:** Establish an objective inactivity threshold (180 days) based on the actual 341-day observation window.
- **Target Leakage Audit:** Conduct a rigorous evaluation to avoid feature-target circularity.
- **Supervised Machine Learning:** Train and evaluate a regularized **Logistic Regression** pipeline with feature scaling.
- **Predictive Scoring & Risk Tiering:** Calculate individual churn probabilities via `predict_proba()` and stratify accounts into **High, Medium, and Low Risk** tiers.
- **Actionable Business Strategy:** Translate model outputs into targeted, ROI-positive retention campaigns.


## 3. Import Libraries & Configure Environment

We import the essential Python scientific stack:
- `pandas` & `numpy` for data manipulation, cleaning, and aggregation
- `matplotlib` & `seaborn` for publication-quality visualizations
- `sklearn` for model preprocessing, train-test splitting, logistic regression, and evaluation
- `pathlib` & `warnings` for robust filesystem navigation and clean outputs


In [ ]:
# Step 4: Import required libraries
import sys
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve
)

warnings.filterwarnings('ignore')

# Set visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.titlesize'] = 14

print("All required libraries successfully imported.")


## 4. Load Dataset

We load the e-commerce transaction dataset from the local `data/raw/` directory or Google Colab upload.
The loader automatically inspects sheet names and loads the raw transactional records into a pandas DataFrame.


In [ ]:
# Step 5: Locate and load the raw dataset
# Supports local project paths, parent directories, and Google Colab environments
possible_paths = [
    Path("data/raw/ecommerce_raw_dataset.xlsx"),
    Path("../data/raw/ecommerce_raw_dataset.xlsx"),
    Path("ecommerce_raw_dataset.xlsx"),
    Path("/content/ecommerce_raw_dataset.xlsx")
]

raw_file_path = None
for p in possible_paths:
    if p.exists():
        raw_file_path = p
        break

if raw_file_path is None:
    raise FileNotFoundError(
        "Could not locate ecommerce_raw_dataset.xlsx. "
        "Please ensure the dataset is placed inside the 'data/raw/' directory or uploaded to Colab."
    )

print(f"Loading raw dataset from: {raw_file_path.resolve()}")

# Load workbook sheet
if raw_file_path.suffix.lower() in ['.xlsx', '.xls']:
    excel_file = pd.ExcelFile(raw_file_path)
    sheet_name = 'Raw_Data' if 'Raw_Data' in excel_file.sheet_names else excel_file.sheet_names[0]
    df_raw = pd.read_excel(raw_file_path, sheet_name=sheet_name)
    print(f"Loaded sheet: '{sheet_name}'")
else:
    df_raw = pd.read_csv(raw_file_path)

print(f"Dataset successfully loaded. Total rows: {len(df_raw):,}, Total columns: {len(df_raw.columns)}")
df_raw.head(5)


## 5. Dataset Understanding & Initial Inspection

Before making any modifications, we perform an automated structural audit:
- Column names and data types
- Number of non-null vs missing values
- Duplicate records
- Distinct counts of Order IDs and Customer IDs


In [ ]:
# Step 6: Dataset inspection and structural audit
print("=" * 60)
print("DATASET INFORMATION & METRICS")
print("=" * 60)
print(f"Shape: {df_raw.shape[0]} rows by {df_raw.shape[1]} columns")
print(f"Duplicate rows: {df_raw.duplicated().sum()}")
if 'Order_ID' in df_raw.columns:
    print(f"Unique Order IDs: {df_raw['Order_ID'].nunique()}")
if 'Customer_ID' in df_raw.columns:
    print(f"Unique Customer IDs (non-null): {df_raw['Customer_ID'].nunique(dropna=True)}")

print("
Column Data Types and Missing Values:")
missing_df = pd.DataFrame({
    'Data Type': df_raw.dtypes,
    'Missing Count': df_raw.isnull().sum(),
    'Missing %': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
missing_df


## 6. Data Cleaning & Hygiene Pipeline

The raw dataset contains several real-world data quality defects:
1. **Duplicate records**: Exact duplicate order rows that artificially inflate sales.
2. **Missing Customer IDs**: 26 rows without customer identity (cannot be linked for churn prediction).
3. **Invalid Calendar Dates**: Impossible dates such as `31-02-2026` or `31-04-2026`.
4. **Negative Quantities**: Erroneous return entries or negative quantities (e.g. `-3.0`).
5. **Formatted Currency Strings**: Revenue strings formatted with currency symbols (`₹50000`, `374814 INR`, commas).

We clean these anomalies systematically and save the cleaned dataset.


In [ ]:
# Step 7: Robust Data Cleaning Pipeline

# 1. Deduplication
df_clean = df_raw.drop_duplicates().copy()
initial_count = len(df_raw)
after_dedup = len(df_clean)

# 2. Drop records missing Customer_ID
df_clean = df_clean.dropna(subset=['Customer_ID']).copy()
after_cust = len(df_clean)

# 3. Standardize and validate Order_Date
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'], errors='coerce')
df_clean = df_clean.dropna(subset=['Order_Date']).copy()
after_date = len(df_clean)

# 4. Clean Quantity (enforce positive values)
df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce')
df_clean = df_clean[df_clean['Quantity'] > 0].copy()
after_qty = len(df_clean)

# 5. Clean Revenue currency strings
def parse_currency(val):
    if pd.isna(val):
        return np.nan
    s = re.sub(r'[^\d.-]', '', str(val).strip())
    try:
        f = float(s)
        return f if f > 0 else np.nan
    except ValueError:
        return np.nan

df_clean['Revenue'] = df_clean['Revenue'].apply(parse_currency)
df_clean = df_clean.dropna(subset=['Revenue']).copy()
after_rev = len(df_clean)

# 6. Clean Profit and categorical text columns
df_clean['Profit'] = pd.to_numeric(df_clean['Profit'], errors='coerce')
for col in ['Product', 'Category', 'Region']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan})

print("=" * 60)
print("DATA CLEANING AUDIT REPORT")
print("=" * 60)
print(f"Raw Input Transactions:          {initial_count:,}")
print(f"Duplicate Rows Dropped:          {initial_count - after_dedup:,}")
print(f"Missing Customer IDs Dropped:    {after_dedup - after_cust:,}")
print(f"Invalid Calendar Dates Dropped:  {after_cust - after_date:,}")
print(f"Negative/Invalid Qty Dropped:    {after_date - after_qty:,}")
print(f"Invalid Revenue Strings Dropped: {after_qty - after_rev:,}")
print(f"Final Cleaned Transactions:      {len(df_clean):,}")
print(f"Valid Unique Customers:          {df_clean['Customer_ID'].nunique():,}")
print(f"Cleaned Date Span:               {df_clean['Order_Date'].min().strftime('%Y-%m-%d')} to {df_clean['Order_Date'].max().strftime('%Y-%m-%d')}")
print(f"Total Net Cleaned Revenue:       INR {df_clean['Revenue'].sum():,.2f}")
print("=" * 60)


## 7. Exploratory Data Analysis (EDA)

We explore the operational patterns in the dataset across 7 key business dimensions:
1. Transaction Revenue Distribution
2. Monthly Revenue Trend (Seasonality)
3. Monthly Order Volume Trend
4. Customer Purchase Frequency Distribution
5. Customer Lifetime Spend Distribution
6. Top 10 High-Value Spenders
7. Average Order Value (AOV) Distribution


In [ ]:
# Step 8.1: Revenue Distribution and Monthly Trends
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# 1. Revenue Distribution
sns.histplot(df_clean['Revenue'], bins=40, kde=True, color='#1f77b4', edgecolor='black', ax=axes[0])
axes[0].set_title("1. Transaction Revenue Distribution", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Order Revenue (INR)", fontsize=11)
axes[0].set_ylabel("Transaction Count", fontsize=11)
axes[0].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"₹{x*1e-3:.0f}K" if x < 1e6 else f"₹{x*1e-6:.1f}M"))

# 2. Monthly Revenue Trend
df_clean['YearMonth'] = df_clean['Order_Date'].dt.to_period('M')
monthly_rev = df_clean.groupby('YearMonth')['Revenue'].sum().reset_index()
monthly_rev['YM_Str'] = monthly_rev['YearMonth'].astype(str)

sns.lineplot(data=monthly_rev, x='YM_Str', y='Revenue', marker='o', color='#2ca02c', linewidth=2.5, ax=axes[1])
axes[1].set_title("2. Monthly Total Revenue Trend (2026)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Month", fontsize=11)
axes[1].set_ylabel("Total Revenue (INR)", fontsize=11)
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, p: f"₹{y*1e-6:.1f}M"))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Step 8.2: Customer Frequency, Top Spenders, and AOV
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# 4. Customer Order Frequency
cust_orders = df_clean.groupby('Customer_ID')['Order_ID'].nunique()
sns.countplot(x=cust_orders, color='#ff7f0e', ax=axes[0])
axes[0].set_title("4. Customer Order Frequency", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Orders Placed per Customer", fontsize=10)
axes[0].set_ylabel("Customer Count", fontsize=10)

# 6. Top 10 Spenders
top10 = df_clean.groupby('Customer_ID')['Revenue'].sum().sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=top10, x='Revenue', y='Customer_ID', palette='magma', ax=axes[1])
axes[1].set_title("6. Top 10 Customers by Revenue", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Total Spend (INR)", fontsize=10)
axes[1].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"₹{x*1e-6:.1f}M"))

# 7. AOV Distribution
cust_aov = df_clean.groupby('Customer_ID')['Revenue'].mean()
sns.histplot(cust_aov, bins=35, kde=True, color='#0ea5e9', ax=axes[2])
axes[2].set_title("7. Average Order Value (AOV) Distribution", fontsize=12, fontweight='bold')
axes[2].set_xlabel("AOV (INR)", fontsize=10)
axes[2].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"₹{x*1e-3:.0f}K"))

plt.tight_layout()
plt.show()


### EDA Findings & Business Interpretation
- **Revenue Distribution:** Highly right-skewed; median order value is INR 18,634 while high-ticket transactions reach up to INR 2.68M.
- **Seasonality:** Peak monthly revenue occurred in April 2026 (INR 14.10M), driven by seasonal promotional campaigns.
- **Single-Order Vulnerability:** 89 customers (21.8% of the cohort) have made only 1 order, representing prime candidates for churn.
- **Revenue Concentration:** The top 10 customers account for INR 14.83M (16.7% of total company revenue), highlighting the necessity of VIP retention protocols.


## 8. Customer-Level Aggregation

To predict whether a customer will churn, the unit of analysis must be shifted from individual orders to unique customer accounts:
$$\text{One Customer} = \text{One Row}$$

We aggregate transaction records by `Customer_ID` to calculate:
- `Order_Count`: Total orders placed (Frequency)
- `Total_Revenue`: Total lifetime spend (Monetary)
- `Avg_Order_Value`: Spend per order
- `First_Purchase`: Date of first interaction
- `Most_Recent_Purchase`: Date of last purchase


In [ ]:
# Step 9: Create customer-level aggregated dataset
cust_df = df_clean.groupby('Customer_ID').agg(
    Order_Count=('Order_ID', 'nunique'),
    Total_Revenue=('Revenue', 'sum'),
    Avg_Order_Value=('Revenue', 'mean'),
    First_Purchase=('Order_Date', 'min'),
    Most_Recent_Purchase=('Order_Date', 'max')
).reset_index()

# Verification of 1 customer = 1 row
assert cust_df['Customer_ID'].is_unique, "Validation Failed: Duplicate Customer IDs detected!"
print(f"Aggregation verified: Exactly {len(cust_df)} rows for {df_clean['Customer_ID'].nunique()} unique customers.")
cust_df.head(5)


## 9. RFM (Recency, Frequency, Monetary) Feature Engineering

RFM analysis is an industry-standard framework for behavioral customer segmentation:
- **Recency ($R$):** Days elapsed between the customer's most recent purchase and the observation reference date (anchor = 2026-12-08).
- **Frequency ($F$):** Total number of distinct orders placed during the observation period.
- **Monetary ($M$):** Total cumulative revenue generated by the customer.
- **Average Order Value ($AOV$):** Monetary spend divided by purchase frequency.


In [ ]:
# Step 10: Calculate RFM metrics
reference_date = df_clean['Order_Date'].max()
print(f"Reference Date for Recency calculation: {reference_date.strftime('%Y-%m-%d')}")

cust_df['Recency'] = (reference_date - cust_df['Most_Recent_Purchase']).dt.days
cust_df['Frequency'] = cust_df['Order_Count']
cust_df['Monetary'] = cust_df['Total_Revenue']

print("Summary of RFM Features:")
cust_df[['Recency', 'Frequency', 'Monetary', 'Avg_Order_Value']].describe().round(2)


## 10. Churn Definition & Churn Status Labeling

### Defining Churn Empirically
In non-contractual e-commerce, churn is not explicitly signaled by cancellation of a subscription; it manifests as prolonged inactivity.
- The dataset spans **341 days** (January 1, 2026 to December 8, 2026).
- A customer with zero purchases for **180 days or more** (~6 months, covering > 52% of the entire observation window) is classified as **Churned**.

$$\text{Churn\_Status} = \begin{cases} 1 & \text{if Recency} \ge 180 \text{ days (Churned)} \\ 0 & \text{if Recency} < 180 \text{ days (Active)} \end{cases}$$


In [ ]:
# Step 11 & 12: Tag Churn Status and Visualize Distribution
CHURN_THRESHOLD_DAYS = 180
cust_df['Churn_Status'] = (cust_df['Recency'] >= CHURN_THRESHOLD_DAYS).astype(int)

churn_counts = cust_df['Churn_Status'].value_counts()
n_churn = churn_counts.get(1, 0)
n_active = churn_counts.get(0, 0)
churn_pct = (n_churn / len(cust_df)) * 100

print("=" * 55)
print("CUSTOMER CHURN DISTRIBUTION (THRESHOLD = 180 DAYS)")
print("=" * 55)
print(f"Active Customers (0):  {n_active:,} ({100 - churn_pct:.2f}%)")
print(f"Churned Customers (1): {n_churn:,} ({churn_pct:.2f}%)")
print(f"Total Customer Base:   {len(cust_df):,}")
print("=" * 55)

# Plotting Churn Status
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.bar(['Active (0)', 'Churned (1)'], [n_active, n_churn], color=['#22c55e', '#ef4444'], edgecolor='black', width=0.5)
ax1.set_title("Customer Churn Counts", fontweight='bold')
ax1.set_ylabel("Number of Customers")
for p in ax1.patches:
    h = p.get_height()
    ax1.annotate(f"{h} ({h/len(cust_df):.1%})", (p.get_x() + p.get_width() / 2., h / 2),
                 ha='center', va='center', color='white', fontweight='bold')

ax2.pie([n_active, n_churn], labels=['Active', 'Churned'], autopct='%1.1f%%',
        colors=['#22c55e', '#ef4444'], startangle=90, wedgeprops=dict(width=0.45, edgecolor='black'))
ax2.set_title("Proportion of Churned Customers", fontweight='bold')
plt.tight_layout()
plt.show()


## 11. Target Leakage Audit

**Target Leakage** occurs when a predictor variable includes information that is directly derived from or identical to the target outcome, giving the model the answer in advance.

### Audit Findings:
1. `Customer_ID`: Unique identity string — must never be used as a predictor.
2. `Recency`: Because `Churn_Status` was defined mathematically as `Recency >= 180`, feeding `Recency` directly into a scaled model causes **artificial 100% accuracy**.
3. **Leakage-Safe Features:** `Frequency`, `Monetary`, and `Avg_Order_Value` reflect purchasing habits and spending capacity independent of the 180-day recency cutoff.


In [ ]:
# Step 13: Audit feature correlations and leakage risks
feature_candidates = ['Recency', 'Frequency', 'Monetary', 'Avg_Order_Value']
corr_matrix = cust_df[feature_candidates + ['Churn_Status']].corr()

print("Feature Correlation with Churn_Status:")
display(corr_matrix[['Churn_Status']].round(4))

print("
Leakage Check Summary:")
print(" - Recency exhibits high correlation (r = 0.7832) because the target was created from it.")
print(" - Using Recency directly creates target leakage in a single-snapshot dataset.")
print(" - Valid Leakage-Safe Feature Set: ['Frequency', 'Monetary', 'Avg_Order_Value']")


## 12. Feature Engineering & Train/Test Split

We isolate the feature matrix $X$ and ground-truth binary target $y$:
- $X = [\text{Frequency, Monetary, Avg\_Order\_Value}]$
- $y = \text{Churn\_Status}$

We perform a stratified **75% / 25% Train-Test split** with `random_state=42` to guarantee exact reproducibility and preserve the 24.2% churn class ratio across both splits.


In [ ]:
# Step 14 & 15: Train-Test Split with Stratification
FEATURE_COLS = ['Frequency', 'Monetary', 'Avg_Order_Value']
X = cust_df[FEATURE_COLS].copy()
y = cust_df['Churn_Status'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(f"Full Dataset Size:     {len(cust_df)} customers")
print(f"Training Subset (75%): {len(X_train)} customers (Churn rate: {y_train.mean():.2%})")
print(f"Testing Subset (25%):  {len(X_test)} customers (Churn rate: {y_test.mean():.2%})")


## 13. Supervised Machine Learning: Logistic Regression

We construct a scikit-learn `Pipeline`:
1. `StandardScaler`: Normalizes monetary values and order counts so large scale differences (e.g. Monetary in tens of thousands vs Frequency in single digits) do not distort gradient convergence.
2. `LogisticRegression`: Fits the sigmoid link function to output calibrated probabilities:
$$P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \sum \beta_i X_i)}}$$


In [ ]:
# Step 16: Train Logistic Regression Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, C=1.0, max_iter=1000))
])

pipeline.fit(X_train, y_train)
print("Logistic Regression pipeline successfully fitted on training data.")

# Inspect learned coefficients
coefficients = pipeline.named_steps['model'].coef_[0]
for feat, coef in zip(FEATURE_COLS, coefficients):
    print(f"Feature: {feat:<18} | Weight (Beta): {coef:>8.4f}")
print(f"Intercept (Beta_0):     {pipeline.named_steps['model'].intercept_[0]:>8.4f}")


## 14. Model Evaluation & Confusion Matrix

We evaluate performance on the held-out **test dataset (N=103)** using industry-standard metrics:
- **Accuracy:** Overall proportion of correct predictions.
- **Precision:** Of all customers predicted to churn, what fraction actually churned?
- **Recall (Sensitivity):** Of all customers who genuinely churned, what fraction did the model detect?
- **Confusion Matrix:** True Negatives (TN), False Positives (FP), False Negatives (FN), True Positives (TP).


In [ ]:
# Step 17 & 18: Generate Test Predictions and Calculate Metrics
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("=" * 55)
print("TEST SET EVALUATION METRICS")
print("=" * 55)
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {prec:.4f} ({prec*100:.2f}%)")
print(f"Recall:    {rec:.4f} ({rec*100:.2f}%)")
print("=" * 55)

# Display Confusion Matrix Heatmap
fig, ax = plt.subplots(figsize=(6, 5))
annot = np.array([
    [f"TN = {tn}\n(Active Retained)", f"FP = {fp}\n(False Alarm)"],
    [f"FN = {fn}\n(Missed At-Risk)", f"TP = {tp}\n(Caught Churn)"]
])
sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', cbar=False, annot_kws={'size': 11, 'weight': 'bold'}, ax=ax,
            xticklabels=['Predicted Active', 'Predicted Churned'],
            yticklabels=['Actual Active', 'Actual Churned'])
ax.set_title("Confusion Matrix (Test Evaluation)", fontweight='bold')
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
plt.show()


## 15. Churn Probability & Customer Risk Stratification

### Risk Level Calibration Logic
In the total cohort, the baseline churn incidence is **24.21%**, with the highest model-predicted probability reaching **51.76%**.
Applying an uncalibrated static threshold of 70% would yield 0 high-risk customers.

We establish calibrated business risk tiers:
- **High Risk ($P \ge 40\%$):** Probability exceeds 1.65x the cohort baseline. Highly vulnerable customers (top ~21%).
- **Medium Risk ($20\% \le P < 40\%$):** Moderate lapse risk around baseline (~35%).
- **Low Risk ($P < 20\%$):** Stable, high-frequency, loyal accounts (~44%).


In [ ]:
# Step 19 & 20: Score All Customers and Assign Risk Tiers
cust_df['Churn_Probability'] = pipeline.predict_proba(cust_df[FEATURE_COLS])[:, 1]
cust_df['Predicted_Churn'] = pipeline.predict(cust_df[FEATURE_COLS])

def assign_tier(p):
    if p >= 0.40:
        return 'High Risk'
    elif p >= 0.20:
        return 'Medium Risk'
    else:
        return 'Low Risk'

cust_df['Risk_Level'] = cust_df['Churn_Probability'].apply(assign_tier)

risk_summary = cust_df['Risk_Level'].value_counts()
print("=" * 55)
print("CUSTOMER BASE RISK DISTRIBUTION")
print("=" * 55)
for level in ['High Risk', 'Medium Risk', 'Low Risk']:
    c = risk_summary.get(level, 0)
    print(f" - {level:<15}: {c:>4} customers ({c/len(cust_df)*100:.2f}%)")
print("=" * 55)


## 16. Top 20 Highest-Risk Customers

We sort the customer base in descending order of churn probability to identify the 20 accounts most vulnerable to defection.


In [ ]:
# Step 21: Top 20 Highest-Risk Customers
display_columns = ['Customer_ID', 'Recency', 'Frequency', 'Monetary', 'Avg_Order_Value', 'Churn_Probability', 'Risk_Level']
risk_table = cust_df[display_columns].sort_values(by='Churn_Probability', ascending=False).reset_index(drop=True)

top20 = risk_table.head(20).copy()
top20_formatted = top20.copy()
top20_formatted['Churn_Probability'] = top20_formatted['Churn_Probability'].map(lambda p: f"{p*100:.2f}%")
top20_formatted['Monetary'] = top20_formatted['Monetary'].map(lambda m: f"₹{m:,.0f}")
top20_formatted['Avg_Order_Value'] = top20_formatted['Avg_Order_Value'].map(lambda a: f"₹{a:,.0f}")

print("TOP 20 HIGHEST-RISK CUSTOMERS:")
top20_formatted


## 17. Risk Level Visualization

We visualize the distribution of accounts across risk tiers and examine how churn probability correlates with purchase frequency.


In [ ]:
# Step 22: Risk tier distributions and behavioral plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

palette = {'High Risk': '#ef4444', 'Medium Risk': '#f59e0b', 'Low Risk': '#10b981'}
order = ['High Risk', 'Medium Risk', 'Low Risk']

# Bar chart of risk counts
bars = ax1.bar(order, [risk_summary.get(k, 0) for k in order], color=[palette[k] for k in order], edgecolor='black', width=0.55)
ax1.set_title("Customer Distribution by Risk Tier", fontweight='bold')
ax1.set_ylabel("Number of Customers")
for b in bars:
    h = b.get_height()
    ax1.annotate(f"{h} ({h/len(cust_df):.1%})", (b.get_x() + b.get_width() / 2., h / 2),
                 ha='center', va='center', color='white', fontweight='bold')

# Scatter: Frequency vs Churn Probability
sns.scatterplot(data=cust_df, x='Frequency', y='Churn_Probability', hue='Risk_Level',
                palette=palette, hue_order=order, alpha=0.8, s=65, ax=ax2)
ax2.axhline(0.40, color='#ef4444', linestyle=':', label='High Risk Cutoff (0.40)')
ax2.axhline(0.20, color='#f59e0b', linestyle=':', label='Medium Risk Cutoff (0.20)')
ax2.set_title("Churn Probability vs Order Frequency", fontweight='bold')
ax2.set_xlabel("Order Frequency (Purchase Count)")
ax2.set_ylabel("Predicted Churn Probability")
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()


## 18. Data-Driven Business Insights

*Note on Scientific Rigor: Insights reflect empirical statistical associations rather than unsubstantiated claims of direct causation.*

1. **Order Frequency is the Dominant Protective Factor:** In our regularized model, purchase frequency exhibited the largest negative coefficient ($-2.7064$). Customers with only 1 order represent over 85% of the High Risk cohort, indicating that cultivating repeat purchase behavior is the single most critical retention lever.
2. **Cumulative Spend Mitigates Risk:** Higher lifetime monetary value is associated with lower churn risk ($\beta = -0.5701$). Customers who have invested significant financial resources show higher brand commitment.
3. **Average Order Value Independence:** Average Order Value displayed negligible predictive influence ($\beta = +0.0096, r = -0.0427$). Large basket sizes during single purchases do not compensate for lack of purchase repetition.
4. **Actionable Segmentation:** Stratifying the 409 accounts isolates 86 High-Risk accounts (21.0%), allowing marketing teams to focus limited retention budgets where churn risk is concentrated.


## 19. Retention Recommendations & Action Plan

| Strategy | Target Segment | Action | Business Reason | Primary Metric |
| :--- | :--- | :--- | :--- | :--- |
| **First-Order Onboarding Sequence** | High-Risk Single Buyers (Frequency = 1, e.g. C182, C416) | Automated 14-day post-purchase sequence with a 15% 2nd-order discount. | Over 85% of high risk accounts made only 1 order; repeat conversion directly reduces churn. | 30-Day Repeat Rate |
| **Lapsed-Customer Re-engagement** | Inactive Customers (Recency 90–150 Days) | Personalized 'We Miss You' emails featuring past product categories and limited-time vouchers. | Re-engages customers before they reach the 180-day permanent churn threshold. | Win-back Conversion Rate |
| **VIP Account Protection Concierge** | High-Value at Risk (Monetary > ₹100,000, e.g. C348, C048) | Direct account manager check-ins, satisfaction calls, and exclusive perks. | Top 10 customers represent 16.7% of total revenue; VIP churn causes severe top-line damage. | VIP Retention Rate & NRR |
| **Mid-Tier Habit Formation** | Medium-Risk Accounts (Frequency 2–3, Risk 20%–40%) | Points-based loyalty tiers and predictive replenishment notifications. | Reinforces repeat purchase cadence and raises customer switching costs. | Quarterly Purchase Frequency |
| **Brand Advocate VIP Circle** | Low-Risk Champions (Frequency ≥ 5, Risk < 20%) | Exclusive pre-access to product drops and high-reward referral programs. | Retains organic loyalty and turns champions into low-cost acquisition drivers. | Net Promoter Score (NPS) |


## 20. Conclusion & Export of Predictions

### Conclusion
By analyzing 1,838 cleaned transactions across 409 customers, this project engineered behavioral RFM features, identified an empirical 180-day churn threshold, audited target leakage, trained a regularized Logistic Regression classifier, and classified customers into risk tiers. The resulting solution empowers e-commerce marketing teams with an interpretable, operationalized early-warning system.

### Export Results
We persist the final ranked risk predictions table to `customer_churn_risk_predictions.csv` for CRM and email automation integration.


In [ ]:
# Step 25 & 26: Export Final Risk Predictions Table
export_filename = "customer_churn_risk_predictions.csv"
risk_table.to_csv(export_filename, index=False)
print(f"Final customer churn risk predictions exported successfully to: {export_filename}")
print(f"Total exported customer records: {len(risk_table):,}")

# Display confirmation of top 5 exported records
risk_table.head(5)
